# 02 · Flare Listesinin Homojenleştirilmesi — Bölüm A
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

**Neden bu defter var?** 01 defterinin ilk bakış grafiğinde Döngü 25'in M+X patlama sayısı, leke sayısı daha düşük olduğu halde Döngü 23'ü açıkça geçiyor. Bunun iki olası sebebi var:

1. **Ölçek farkı:** NOAA'nın operasyonel sınıfları GOES 8–15 döneminde 0.7 çarpanı içeriyordu, GOES-16 ve sonrası içermiyor. Eski patlamalar yaklaşık %30 zayıf görünüyor.
2. **Çift sayım:** HEK aynı patlamayı farklı kaynaklardan (`frm_name`) birden fazla kez kaydetmiş olabilir. 01 defteri yalnızca birebir aynı zamanlı kayıtları silebildi.

Bu defter önce HEK listesini teşhis eder, sonra NOAA NCEI'nin bilimsel kalite flare özet dosyalarını indirip içlerini inceler.

**Bölüm A** (bu defter): teşhis + indirme + dosya yapısını inceleme. Çıktılara göre **Bölüm B**'de dosyalar okunup tek bir homojen liste oluşturulacak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, re, hashlib, time, datetime as dt
import requests, numpy as np, pandas as pd

KOK = Path('/content/drive/MyDrive/gunes_tezi')
HAM, ISLENMIS, SEKIL = KOK/'data'/'raw', KOK/'data'/'processed', KOK/'figures'
(HAM/'ncei'/'flsum').mkdir(parents=True, exist_ok=True)

KAYIT = KOK/'data'/'indirme_kaydi.json'
_kayit = json.loads(KAYIT.read_text()) if KAYIT.exists() else {}
def indir(url, hedef, deneme=4):
    hedef = Path(hedef)
    if hedef.exists() and hedef.stat().st_size > 0:
        return hedef
    for i in range(deneme):
        try:
            r = requests.get(url, timeout=300); r.raise_for_status()
            hedef.write_bytes(r.content)
            _kayit[str(hedef.relative_to(KOK))] = {
                'url': url,
                'indirme_utc': dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                'bayt': len(r.content), 'sha256': hashlib.sha256(r.content).hexdigest()}
            KAYIT.write_text(json.dumps(_kayit, indent=2, ensure_ascii=False))
            return hedef
        except Exception as e:
            print(f'  ! {e}'); time.sleep(5*(i+1))
    raise RuntimeError(f'İndirilemedi: {url}')
print('Hazır')

## 1 · HEK listesinin teşhisi
Her yıl için hangi kaynaktan (`frm_name`) kaç kayıt geldiğine bakıyoruz. Belirli yıllarda ikinci bir kaynak beliriyorsa çift sayım var demektir. Ayrıca 30 dakika içinde aynı aktif bölgeden gelen, aynı sınıftaki kayıtları "olası tekrar" olarak sayıyoruz.

In [ ]:
fl = pd.read_parquet(ISLENMIS/'flare_hek.parquet')
fl['yil'] = fl.event_peaktime.dt.year

print('Kaynaklara göre toplam kayıt:')
print(fl.frm_name.value_counts(dropna=False).to_string())

tablo = fl.pivot_table(index='yil', columns='frm_name', values='fl_goescls', aggfunc='size', fill_value=0)
print('\nYıl × kaynak (kayıt sayısı):')
print(tablo.to_string())

# Olası tekrarlar: aynı sınıf harfi, aralarında 30 dk'dan az fark
f = fl.sort_values('event_peaktime').reset_index(drop=True)
fark = f.event_peaktime.diff().dt.total_seconds().div(60)
ayni = (fark < 30) & (f.sinif == f.sinif.shift()) & (f.frm_name != f.frm_name.shift())
f['olasi_tekrar'] = ayni
print('\nFarklı kaynaktan, 30 dk içinde, aynı sınıfta olası tekrar sayısı (yıllık):')
print(f.groupby('yil').olasi_tekrar.sum().astype(int).to_string())

print('\nŞüpheli kayıt kontrolü — 2024-05-08 civarındaki X sınıfı kayıtlar:')
print(f[(f.event_peaktime.between('2024-05-07','2024-05-10')) & (f.sinif=='X')]
      [['event_starttime','event_peaktime','fl_goescls','ar_noaanum','frm_name']].to_string(index=False))

## 2 · NCEI bilimsel kalite flare özetlerini indir
01 defterindeki tarama, her uydu için görev boyunca tek bir bilimsel kalite dosyası olduğunu gösterdi:
- **GOES 8–15:** `sci_xrsf-l2-flsum_gNN_sYYYYMMDD_eYYYYMMDD_v1-0-0.nc`
- **GOES 16–19:** `xrsf-l2-flsum_science/sci_xrsf-l2-flsum_gNN_s..._e..._v2-2-2.nc`

Yıllık dosyaları ve operasyonel `dn_` dosyalarını almıyoruz, yalnızca görev boyu bilimsel dosyaları indiriyoruz.

In [ ]:
bulunan = json.loads((HAM/'ncei'/'flsum_bulunanlar.json').read_text())
tum = sorted({u for liste in bulunan.values() for u in liste})
desen = re.compile(r'/sci_xrsf-l2-flsum_g(\d{2})_s\d{8}_e\d{8}_v[\d-]+\.nc$')
secilen = {}
for u in tum:
    m = desen.search(u)
    if m:
        secilen.setdefault(int(m.group(1)), []).append(u)

for uydu in sorted(secilen):
    for u in secilen[uydu]:
        ad = u.rsplit('/', 1)[1]
        p = indir(u, HAM/'ncei'/'flsum'/ad)
        print(f'GOES-{uydu:02d}  {ad}  ({p.stat().st_size/1e6:.1f} MB)')

## 3 · Dosyaların içini incele
Her dosya ailesinden birer örneği açıp değişken adlarını, boyutlarını ve ilk satırlarını yazdırıyoruz. **Bu hücrenin çıktısının tamamını gönder.** Bölüm B'deki okuma kodu bu yapıya göre yazılacak.

In [ ]:
import xarray as xr
dosyalar = sorted((HAM/'ncei'/'flsum').glob('*.nc'))
print('İndirilen dosyalar:'); [print('  ', d.name) for d in dosyalar]

def incele(p):
    print('\n' + '=' * 100); print(p.name); print('=' * 100)
    ds = xr.open_dataset(p, decode_times=True)
    print(ds)
    print('\n--- Değişkenler ---')
    for v in ds.variables:
        a = ds[v].attrs
        print(f'{v:28s} {str(ds[v].dtype):16s} {str(ds[v].dims):20s} '
              f'{a.get("long_name", a.get("description", ""))[:60]}')
    tek_boyut = [v for v in ds.data_vars if ds[v].ndim == 1]
    print('\n--- İlk 8 satır ---')
    print(ds[tek_boyut].to_dataframe().head(8).to_string())
    for v in tek_boyut:
        if ds[v].dtype.kind in 'OSU':
            print(f'\n{v} benzersiz değerlerden örnekler:', pd.Series(ds[v].values).astype(str).unique()[:10])
    ds.close()

eski = [d for d in dosyalar if re.search(r'_g(0[89]|1[0-5])_', d.name)]
yeni = [d for d in dosyalar if re.search(r'_g1[6-9]_', d.name)]
if eski: incele(eski[-1])
if yeni: incele(yeni[0])